## 1. Import Library

In [1]:
#202431013_Tashfia Wahdah Fuady
import requests
import csv
import pandas as pd
import re
from bs4 import BeautifulSoup

### 2. Mengambil halaman web dan mengecek status code

In [2]:
#202431013_Tashfia Wahdah Fuady
url = "https://id.wikipedia.org/wiki/Daftar_sekolah_dasar_negeri_di_Indonesia"

headers = { "User-Agent": "Mozilla/5.0" }

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print("Status Code:", response.status_code)

Status Code: 200


### 3. Mengubah HTML menjadi objek BeautifulSoup

In [3]:
#202431013_Tashfia Wahdah Fuady
soup = BeautifulSoup(response.text, "html.parser")

print("HTML berhasil diubah menjadi BeautifulSoup")
print("Judul halaman:", soup.title.get_text(strip=True))

HTML berhasil diubah menjadi BeautifulSoup
Judul halaman: Daftar sekolah dasar negeri di Indonesia - Wikipedia bahasa Indonesia, ensiklopedia bebas


### 4. Mencari seluruh blok data menggunakan find_all()

In [4]:
#202431013_Tashfia Wahdah Fuady
# Mencari seluruh tabel
tables = soup.find_all("table")

print("Jumlah seluruh tabel:", len(tables))

data_blocks = []

for table in tables:

    rows = table.find_all("tr")
    heading = table.find_previous("h2")
    province = "Tidak diketahui"

    if heading:
        headline = heading.find("span", class_="mw-headline")

        if headline:
            province = headline.get_text(strip=True)

    for row in rows:
        cells = row.find_all(["td", "th"],recursive=False)

        # Memastikan jumlah kolom mencukupi
        if len(cells) < 15:
            continue

        # Mengambil nomor urut
        nomor = cells[0].get_text(strip=True)

        # Mengambil nama kabupaten/kota
        wilayah = cells[1].get_text(" ", strip=True)

        # Memastikan baris merupakan data wilayah
        if nomor.isdigit() and re.match(r"^(Kab\.|Kabupaten|Kota)(\s|$)", wilayah):

            data_blocks.append({"province": province, "row": row})

print("Jumlah blok data kabupaten/kota:", len(data_blocks))

Jumlah seluruh tabel: 37
Jumlah blok data kabupaten/kota: 481


In [5]:
#202431013_Tashfia Wahdah Fuady
if len(data_blocks) >= 200:
    print("Jumlah data memenuhi syarat minimal 200 baris.")
else:
    print("Jumlah data belum memenuhi syarat minimal 200 baris.")

Jumlah data memenuhi syarat minimal 200 baris.


### 5. Mengambil setiap atribut dan menyimpannya ke list result

In [6]:
#202431013_Tashfia Wahdah Fuady
# Fungsi untuk membersihkan angka
def clean_number(text):

    # Menghapus tanda referensi seperti [1]
    text = re.sub(r"\[\d+\]", "", text)

    # Menghapus tanda titik, koma, dan karakter nonangka
    number = re.sub(r"\D", "", text)

    if number:
        return int(number)

    return None

In [7]:
#202431013_Tashfia Wahdah Fuady
result = []

for item in data_blocks:

    row = item["row"]
    province = item["province"]

    cells = row.find_all(["td", "th"], recursive=False)

    # Mengambil data kabupaten/kota
    wilayah = cells[1].get_text(" ", strip=True)

    # Mengambil atribut SD
    sd_negeri = clean_number(cells[2].get_text(strip=True))
    sd_swasta = clean_number(cells[3].get_text(strip=True))
    sd_jumlah = clean_number(cells[4].get_text(strip=True))

    # Mengambil atribut SMP
    smp_negeri = clean_number(cells[5].get_text(strip=True))
    smp_swasta = clean_number(cells[6].get_text(strip=True))
    smp_jumlah = clean_number(cells[7].get_text(strip=True))

    # Mengambil atribut SMA
    sma_negeri = clean_number(cells[8].get_text(strip=True))
    sma_swasta = clean_number(cells[9].get_text(strip=True))
    sma_jumlah = clean_number(cells[10].get_text(strip=True))

    # Mengambil atribut SMK
    smk_negeri = clean_number(cells[11].get_text(strip=True))
    smk_swasta = clean_number(cells[12].get_text(strip=True))
    smk_jumlah = clean_number(cells[13].get_text(strip=True))

    # Mengambil total seluruh sekolah
    total_sekolah = clean_number(
        cells[14].get_text(strip=True)
    )

    # Menyimpan hasil ke dalam list
    result.append({
        "Provinsi": province,
        "Kabupaten_Kota": wilayah,
        "SD_Negeri": sd_negeri,
        "SD_Swasta": sd_swasta,
        "Jumlah_SD": sd_jumlah,
        "SMP_Negeri": smp_negeri,
        "SMP_Swasta": smp_swasta,
        "Jumlah_SMP": smp_jumlah,
        "SMA_Negeri": sma_negeri,
        "SMA_Swasta": sma_swasta,
        "Jumlah_SMA": sma_jumlah,
        "SMK_Negeri": smk_negeri,
        "SMK_Swasta": smk_swasta,
        "Jumlah_SMK": smk_jumlah,
        "Total_Seluruh_Sekolah": total_sekolah
    })

print("Jumlah data berhasil diambil:", len(result))

Jumlah data berhasil diambil: 481


### 6. Menampilkan hasil scraping dengan perulangan for

In [8]:
#202431013_Tashfia Wahdah Fuady
for i, data in enumerate(result[:10], start=1):

    print(f"\nData ke-{i}")

    for key, value in data.items():

        print(f"{key}: {value}")


Data ke-1
Provinsi: Tidak diketahui
Kabupaten_Kota: Kab. Aceh Besar
SD_Negeri: 247
SD_Swasta: 15
Jumlah_SD: 262
SMP_Negeri: 60
SMP_Swasta: 48
Jumlah_SMP: 108
SMA_Negeri: 34
SMA_Swasta: 34
Jumlah_SMA: 68
SMK_Negeri: 8
SMK_Swasta: 3
Jumlah_SMK: 11
Total_Seluruh_Sekolah: 449

Data ke-2
Provinsi: Tidak diketahui
Kabupaten_Kota: Kab. Pidie
SD_Negeri: 325
SD_Swasta: 14
Jumlah_SD: 339
SMP_Negeri: 65
SMP_Swasta: 35
Jumlah_SMP: 100
SMA_Negeri: 32
SMA_Swasta: 19
Jumlah_SMA: 51
SMK_Negeri: 7
SMK_Swasta: 7
Jumlah_SMK: 14
Total_Seluruh_Sekolah: 504

Data ke-3
Provinsi: Tidak diketahui
Kabupaten_Kota: Kab. Aceh Utara
SD_Negeri: 395
SD_Swasta: 17
Jumlah_SD: 412
SMP_Negeri: 107
SMP_Swasta: 85
Jumlah_SMP: 192
SMA_Negeri: 43
SMA_Swasta: 39
Jumlah_SMA: 82
SMK_Negeri: 14
SMK_Swasta: 10
Jumlah_SMK: 24
Total_Seluruh_Sekolah: 710

Data ke-4
Provinsi: Tidak diketahui
Kabupaten_Kota: Kab. Aceh Timur
SD_Negeri: 317
SD_Swasta: 18
Jumlah_SD: 335
SMP_Negeri: 82
SMP_Swasta: 30
Jumlah_SMP: 112
SMA_Negeri: 32
SMA_Sw

### 7. Menyimpan hasil scraping ke file CSV

In [9]:
#202431013_Tashfia Wahdah Fuady
nama_file = "data_sekolah_indonesia.csv"

fieldnames = [
    "Provinsi",
    "Kabupaten_Kota",
    "SD_Negeri",
    "SD_Swasta",
    "Jumlah_SD",
    "SMP_Negeri",
    "SMP_Swasta",
    "Jumlah_SMP",
    "SMA_Negeri",
    "SMA_Swasta",
    "Jumlah_SMA",
    "SMK_Negeri",
    "SMK_Swasta",
    "Jumlah_SMK",
    "Total_Seluruh_Sekolah"
]

with open(
    nama_file,
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames
    )

    writer.writeheader()

    for data in result:
        writer.writerow(data)

print("Data berhasil disimpan ke:", nama_file)

Data berhasil disimpan ke: data_sekolah_indonesia.csv


### 8. Membaca file CSV menggunakan pandas

In [10]:
#202431013_Tashfia Wahdah Fuady
df = pd.read_csv(nama_file)
display(df.head(10))

print("Ukuran dataset:", df.shape)
print("\nInformasi Dataset:")
df.info()

,Provinsi,Kabupaten_Kota,SD_Negeri,SD_Swasta,Jumlah_SD,SMP_Negeri,SMP_Swasta,Jumlah_SMP,SMA_Negeri,SMA_Swasta,Jumlah_SMA,SMK_Negeri,SMK_Swasta,Jumlah_SMK,Total_Seluruh_Sekolah
0,Tidak diketahui,Kab. Aceh Besar,247.0,15,262,60,48,108,34,34,68,8,3,11,449
1,Tidak diketahui,Kab. Pidie,325.0,14,339,65,35,100,32,19,51,7,7,14,504
2,Tidak diketahui,Kab. Aceh Utara,395.0,17,412,107,85,192,43,39,82,14,10,24,710
3,Tidak diketahui,Kab. Aceh Timur,317.0,18,335,82,30,112,32,16,48,14,1,15,510
4,Tidak diketahui,Kab. Aceh Tengah,207.0,19,226,50,21,71,21,15,36,4,1,5,338
5,Tidak diketahui,Kab. Aceh Barat,174.0,14,188,52,19,71,20,9,29,10,1,11,299
6,Tidak diketahui,Kab. Aceh Selatan,228.0,11,239,56,26,82,32,14,46,10,1,11,378
7,Tidak diketahui,Kab. Aceh Tenggara,155.0,47,202,40,52,92,23,22,45,6,10,16,355
8,Tidak diketahui,Kab. Simeulue,119.0,9,128,45,12,57,26,5,31,8,0,8,224
9,Tidak diketahui,Kab. Bireuen,283.0,8,291,73,47,120,31,25,56,8,4,12,479


Ukuran dataset: (481, 15)

Informasi Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Provinsi               481 non-null    str    
 1   Kabupaten_Kota         481 non-null    str    
 2   SD_Negeri              480 non-null    float64
 3   SD_Swasta              481 non-null    int64  
 4   Jumlah_SD              481 non-null    int64  
 5   SMP_Negeri             481 non-null    int64  
 6   SMP_Swasta             481 non-null    int64  
 7   Jumlah_SMP             481 non-null    int64  
 8   SMA_Negeri             481 non-null    int64  
 9   SMA_Swasta             481 non-null    int64  
 10  Jumlah_SMA             481 non-null    int64  
 11  SMK_Negeri             481 non-null    int64  
 12  SMK_Swasta             481 non-null    int64  
 13  Jumlah_SMK             481 non-null    int64  
 14  Total_Seluruh_Sekolah  

In [11]:
display(df)

,Provinsi,Kabupaten_Kota,SD_Negeri,SD_Swasta,Jumlah_SD,SMP_Negeri,SMP_Swasta,Jumlah_SMP,SMA_Negeri,SMA_Swasta,Jumlah_SMA,SMK_Negeri,SMK_Swasta,Jumlah_SMK,Total_Seluruh_Sekolah
0,Tidak diketahui,Kab. Aceh Besar,247.0,15,262,60,48,108,34,34,68,8,3,11,449
1,Tidak diketahui,Kab. Pidie,325.0,14,339,65,35,100,32,19,51,7,7,14,504
2,Tidak diketahui,Kab. Aceh Utara,395.0,17,412,107,85,192,43,39,82,14,10,24,710
3,Tidak diketahui,Kab. Aceh Timur,317.0,18,335,82,30,112,32,16,48,14,1,15,510
4,Tidak diketahui,Kab. Aceh Tengah,207.0,19,226,50,21,71,21,15,36,4,1,5,338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476,Tidak diketahui,Kab. Tambrauw,33.0,23,56,12,3,15,4,1,5,3,1,4,80
477,Tidak diketahui,Kab. Maybrat,17.0,42,59,11,2,13,4,0,4,1,0,1,77
478,Tidak diketahui,Kab. Pegunungan Arfak,47.0,13,60,13,1,14,7,0,7,2,0,2,83
479,Tidak diketahui,Kab. Manokwari Selatan,22.0,7,29,5,3,8,4,1,5,0,1,1,43


### 9. Memastikan tipe data numerik dan menyimpan dataset final

In [12]:
#202431013_Tashfia Wahdah Fuady
# Menentukan kolom numerik
kolom_numerik = [
    "SD_Negeri",
    "SD_Swasta",
    "Jumlah_SD",
    "SMP_Negeri",
    "SMP_Swasta",
    "Jumlah_SMP",
    "SMA_Negeri",
    "SMA_Swasta",
    "Jumlah_SMA",
    "SMK_Negeri",
    "SMK_Swasta",
    "Jumlah_SMK",
    "Total_Seluruh_Sekolah"
]

# Mengubah kolom menjadi tipe numerik
for kolom in kolom_numerik:

    df[kolom] = pd.to_numeric(
        df[kolom],
        errors="coerce"
    ).astype("Int64")

# Memastikan kolom kategori bertipe teks
df["Provinsi"] = df["Provinsi"].astype("string")
df["Kabupaten_Kota"] = df["Kabupaten_Kota"].astype("string")

# Menampilkan tipe data setelah konversi
print("Tipe Data Setelah Konversi:")
print(df.dtypes)

Tipe Data Setelah Konversi:
Provinsi                 string
Kabupaten_Kota           string
SD_Negeri                 Int64
SD_Swasta                 Int64
Jumlah_SD                 Int64
SMP_Negeri                Int64
SMP_Swasta                Int64
Jumlah_SMP                Int64
SMA_Negeri                Int64
SMA_Swasta                Int64
Jumlah_SMA                Int64
SMK_Negeri                Int64
SMK_Swasta                Int64
Jumlah_SMK                Int64
Total_Seluruh_Sekolah     Int64
dtype: object


In [13]:
#202431013_Tashfia Wahdah Fuady
df.to_csv(
    "data_sekolah_indonesia_final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset final berhasil disimpan!")

Dataset final berhasil disimpan!


## Pemeriksaan Akhir

In [14]:
#202431013_Tashfia Wahdah Fuady
print("===== PEMERIKSAAN DATASET =====")

print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))

print("\nJumlah data duplikat:", df.duplicated(
    subset=["Provinsi", "Kabupaten_Kota"]
).sum())

print("\nJumlah missing value:")
print(df.isnull().sum())

print("\nTipe data:")
print(df.dtypes)

if len(df) >= 200:
    print("\nDataset memenuhi syarat minimal 200 data.")
else:
    print("\nDataset belum memenuhi syarat minimal 200 data.")

===== PEMERIKSAAN DATASET =====
Jumlah baris: 481
Jumlah kolom: 15

Jumlah data duplikat: 8

Jumlah missing value:
Provinsi                 0
Kabupaten_Kota           0
SD_Negeri                1
SD_Swasta                0
Jumlah_SD                0
SMP_Negeri               0
SMP_Swasta               0
Jumlah_SMP               0
SMA_Negeri               0
SMA_Swasta               0
Jumlah_SMA               0
SMK_Negeri               0
SMK_Swasta               0
Jumlah_SMK               0
Total_Seluruh_Sekolah    0
dtype: int64

Tipe data:
Provinsi                 string
Kabupaten_Kota           string
SD_Negeri                 Int64
SD_Swasta                 Int64
Jumlah_SD                 Int64
SMP_Negeri                Int64
SMP_Swasta                Int64
Jumlah_SMP                Int64
SMA_Negeri                Int64
SMA_Swasta                Int64
Jumlah_SMA                Int64
SMK_Negeri                Int64
SMK_Swasta                Int64
Jumlah_SMK                Int64
Total_S